# Econometric Forecast Engine: Interactive Diagnostics & Forecasting

This notebook demonstrates exploratory time-series diagnostics, unit-root tests (`urca`), state-space models (`forecast`), and vector autoregressions (`vars`) integrating Python and R via `rpy2`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from app.ingestion.market_loader import MarketDataLoader
from app.ingestion.preprocessor import TimeSeriesPreprocessor
from app.bridge.r_forecaster import RForecaster

# Enable R cell magic
%load_ext rpy2.ipython
print("Python scientific stack and rpy2 bridge initialized.")

## 1. Market Data Ingestion & Returns Computation

In [ ]:
loader = MarketDataLoader()
spy_prices = loader.load_ticker("SPY", start_date="2021-01-01")
returns = TimeSeriesPreprocessor.compute_log_returns(spy_prices)

print(f"Loaded {len(spy_prices)} price observations for SPY.")
print(f"Mean daily return: {returns.mean():.4%}, Volatility: {returns.std():.4%}")

## 2. Unit Root & Stationarity Tests (ADF vs. KPSS)

In [ ]:
forecaster = RForecaster()
price_stationarity = forecaster.test_stationarity(spy_prices, type="trend")
returns_stationarity = forecaster.test_stationarity(returns, type="drift")

print("Raw Prices Verdict:", price_stationarity.consensus_verdict)
print("Log-Returns Verdict:", returns_stationarity.consensus_verdict)

## 3. Pure R Cell Magic: State-Space Modeling with `forecast`

In [ ]:
%%R -i spy_prices
library(forecast)
spy_ts <- ts(spy_prices, frequency = 7)

# Automatic Hyndman-Khandakar ARIMA selection
fit_arima <- auto.arima(spy_ts, stepwise = TRUE)
print(summary(fit_arima))

# 14-step ahead forecast with 80% and 95% intervals
fc <- forecast(fit_arima, h = 14)
print(fc)

## 4. Vector Autoregression (VAR) & Granger Causality

In [ ]:
multi_df = loader.load_multi_asset(["SPY", "TLT", "GLD"], start_date="2021-01-01")
log_returns_df = TimeSeriesPreprocessor.compute_log_returns(multi_df)

var_res = forecaster.fit_multivariate_var(log_returns_df, max_lag=4, horizon=10)
print(f"Selected VAR Lag: p={var_res.lag_order}, Stability: {var_res.is_stable}")
for var_name, gc in var_res.granger_causality.items():
    print(f"{var_name}: p={gc['p_value']:.4f} -> {gc['interpretation']}")